# RAG Document Assistant

End-to-end pipeline for intelligent document retrieval and question answering over regulatory banking documents.

**Stack:** Python · LangChain · FAISS · OpenAI · FastAPI · MLflow

In [ ]:
!pip install langchain langchain-community langchain-openai openai faiss-cpu \
             tiktoken python-dotenv mlflow fastapi uvicorn pypdf -q

In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')
from dotenv import load_dotenv
load_dotenv()

CONFIG = {
    "embedding_model": "text-embedding-ada-002",
    "llm_model": "gpt-3.5-turbo",
    "chunk_size": 500,
    "chunk_overlap": 50,
    "top_k": 4,
    "faiss_index_path": "faiss_index",
    "mlflow_experiment": "RAG-Document-Assistant",
}

## Document Ingestion

In [ ]:
import os

DOCS_DIR = "./sample_docs"
os.makedirs(DOCS_DIR, exist_ok=True)

sample_documents = {
    "aml_policy.txt": """
    Anti-Money Laundering (AML) Policy - Version 3.2

    Section 1: Customer Due Diligence (CDD)
    All customers must be verified through Know Your Customer (KYC) procedures before
    account opening. Enhanced Due Diligence (EDD) is required for high-risk customers
    including Politically Exposed Persons (PEPs) and customers from high-risk jurisdictions.

    Section 2: Transaction Monitoring
    Automated systems monitor all transactions above $10,000 for suspicious activity.
    Structuring transactions to avoid reporting thresholds is strictly prohibited.
    All suspicious activity must be reported via Suspicious Activity Reports (SARs) within 30 days.

    Section 3: Sanctions Screening
    Real-time screening against OFAC, UN, and EU sanctions lists is required for all
    transactions and customer onboarding. Any matches trigger an automatic hold and
    escalation to the compliance team within 24 hours.
    """,

    "credit_risk_policy.txt": """
    Credit Risk Assessment Policy - Retail Banking

    Section 1: Credit Scoring Model
    The bank uses a proprietary credit scoring model combining FICO scores, debt-to-income
    ratios, employment history, and behavioral banking data. Minimum credit score for
    personal loans is 650; for mortgages, 700.

    Section 2: Loan Approval Thresholds
    Personal loans up to $25,000 may be approved by branch managers.
    Loans between $25,000 and $100,000 require regional credit committee approval.
    Any loan exceeding $100,000 must be reviewed by the Chief Risk Officer.

    Section 3: Fraud Detection
    Machine learning models analyze transaction patterns to detect fraud in real-time.
    Transactions with risk scores above 0.85 are automatically blocked and flagged for review.
    False positive rate is maintained below 2% to minimize customer friction.
    """,

    "data_privacy_policy.txt": """
    Customer Data Privacy Policy - GDPR & CCPA Compliance

    Section 1: Data Collection
    We collect only data necessary for banking services including identity verification,
    transaction processing, and regulatory compliance. Customers are informed of all
    data collection at the point of collection via clear consent forms.

    Section 2: Data Retention
    Transaction records are retained for 7 years per regulatory requirements.
    Customer PII is anonymized after account closure plus a 2-year retention period.
    Customers may request data deletion for non-regulatory data at any time.

    Section 3: Data Access Rights
    Customers have the right to access, correct, and export their personal data.
    Data access requests must be fulfilled within 30 days as required by GDPR Article 15.
    """
}

for filename, content in sample_documents.items():
    with open(os.path.join(DOCS_DIR, filename), 'w') as f:
        f.write(content)

print(f"Loaded {len(sample_documents)} documents")

In [ ]:
from langchain.document_loaders import DirectoryLoader, TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

loader = DirectoryLoader(DOCS_DIR, glob="**/*.txt", loader_cls=TextLoader, show_progress=True)
raw_documents = loader.load()

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CONFIG["chunk_size"],
    chunk_overlap=CONFIG["chunk_overlap"],
    separators=["\n\n", "\n", ". ", " ", ""]
)
chunks = text_splitter.split_documents(raw_documents)

print(f"{len(raw_documents)} documents → {len(chunks)} chunks")

## Build FAISS Vector Store

In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
import time

embeddings = OpenAIEmbeddings(model=CONFIG["embedding_model"])

start = time.time()
vectorstore = FAISS.from_documents(documents=chunks, embedding=embeddings)
vectorstore.save_local(CONFIG["faiss_index_path"])

print(f"Index built in {time.time() - start:.1f}s — {vectorstore.index.ntotal} vectors stored")

In [ ]:
# To reload an existing index on subsequent runs:
# vectorstore = FAISS.load_local(CONFIG["faiss_index_path"], embeddings, allow_dangerous_deserialization=True)

retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": CONFIG["top_k"]}
)

## RAG Chain

In [ ]:
from langchain_openai import ChatOpenAI
from langchain.prompts import ChatPromptTemplate
from langchain.schema.runnable import RunnablePassthrough
from langchain.schema.output_parser import StrOutputParser

PROMPT = """
You are a compliance and regulatory document assistant for a banking institution.
Answer questions using ONLY the context provided below.
If the context is insufficient, say so explicitly. Cite the relevant section when possible.

Context:
{context}

Question: {question}

Answer:
"""

prompt = ChatPromptTemplate.from_template(PROMPT)
llm = ChatOpenAI(model=CONFIG["llm_model"], temperature=0, max_tokens=500)

def format_docs(docs):
    return "\n\n".join(
        f"[{doc.metadata.get('source', 'unknown').split('/')[-1]}]\n{doc.page_content}"
        for doc in docs
    )

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

## Query the System

In [ ]:
def ask(question: str, show_sources: bool = True):
    print(f"\nQ: {question}")
    print("-" * 60)

    if show_sources:
        docs = retriever.invoke(question)
        for i, doc in enumerate(docs, 1):
            src = doc.metadata.get('source', '').split('/')[-1]
            print(f"  [{i}] {src}: {doc.page_content[:80].strip()}...")
        print()

    answer = rag_chain.invoke(question)
    print(answer)
    return answer

In [ ]:
ask("What is the threshold for reporting suspicious transactions?")

In [ ]:
ask("What is the minimum credit score required for a mortgage?")

In [ ]:
ask("How long does the bank retain transaction records?")

In [ ]:
# Out-of-scope — tests grounding / hallucination prevention
ask("What is the bank's investment strategy for 2025?")

## Evaluation with MLflow

In [ ]:
import mlflow

EVAL_DATASET = [
    {
        "question": "What is the threshold for reporting suspicious transactions?",
        "expected_keywords": ["10,000", "SAR", "30 days"],
        "expected_source": "aml_policy.txt"
    },
    {
        "question": "What is the minimum credit score for a personal loan?",
        "expected_keywords": ["650", "personal loan"],
        "expected_source": "credit_risk_policy.txt"
    },
    {
        "question": "How long are transaction records retained?",
        "expected_keywords": ["7 years", "regulatory"],
        "expected_source": "data_privacy_policy.txt"
    },
    {
        "question": "Who approves loans above $100,000?",
        "expected_keywords": ["Chief Risk Officer"],
        "expected_source": "credit_risk_policy.txt"
    },
]

def evaluate_retrieval(dataset):
    results = []
    for item in dataset:
        docs = retriever.invoke(item["question"])
        retrieved_sources = [d.metadata.get("source", "") for d in docs]
        retrieved_text = " ".join(d.page_content for d in docs).lower()

        source_hit = any(item["expected_source"] in s for s in retrieved_sources)
        kw_hits = sum(1 for kw in item["expected_keywords"] if kw.lower() in retrieved_text)
        kw_precision = kw_hits / len(item["expected_keywords"])

        results.append({"source_hit": source_hit, "kw_precision": kw_precision})
        status = "PASS" if source_hit else "FAIL"
        print(f"[{status}] {item['question'][:55]}... | keywords: {kw_precision:.0%}")

    return {
        "source_precision": sum(r["source_hit"] for r in results) / len(results),
        "avg_keyword_precision": sum(r["kw_precision"] for r in results) / len(results),
    }

metrics = evaluate_retrieval(EVAL_DATASET)
print(f"\nSource precision    : {metrics['source_precision']:.0%}")
print(f"Keyword precision   : {metrics['avg_keyword_precision']:.0%}")

In [ ]:
mlflow.set_experiment(CONFIG["mlflow_experiment"])

with mlflow.start_run():
    mlflow.log_params({
        "embedding_model": CONFIG["embedding_model"],
        "llm_model": CONFIG["llm_model"],
        "chunk_size": CONFIG["chunk_size"],
        "chunk_overlap": CONFIG["chunk_overlap"],
        "top_k": CONFIG["top_k"],
        "n_chunks": len(chunks),
    })
    mlflow.log_metrics({
        "source_precision": metrics["source_precision"],
        "avg_keyword_precision": metrics["avg_keyword_precision"],
    })

print("Run logged. View with: mlflow ui")

## FastAPI Serving Layer

Save the cell below as `api.py` and run with `uvicorn api:app --reload`

In [ ]:
api_code = '''
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain.prompts import ChatPromptTemplate
from langchain.schema.runnable import RunnablePassthrough
from langchain.schema.output_parser import StrOutputParser
import time

app = FastAPI(title="RAG Document Assistant", version="1.0.0")

embeddings = OpenAIEmbeddings(model="text-embedding-ada-002")
vectorstore = FAISS.load_local("faiss_index", embeddings, allow_dangerous_deserialization=True)
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

PROMPT = """
You are a compliance document assistant. Answer ONLY from the context below.
If context is insufficient, say so. Cite sections when possible.

Context: {context}
Question: {question}
Answer:
"""
prompt = ChatPromptTemplate.from_template(PROMPT)
llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)

def format_docs(docs):
    return "\\n\\n".join(d.page_content for d in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt | llm | StrOutputParser()
)

class QueryRequest(BaseModel):
    question: str
    include_sources: bool = False

class QueryResponse(BaseModel):
    answer: str
    sources: list[str] = []
    latency_ms: float

@app.get("/health")
def health():
    return {"status": "ok", "vectors": vectorstore.index.ntotal}

@app.post("/query", response_model=QueryResponse)
def query(req: QueryRequest):
    if not req.question.strip():
        raise HTTPException(status_code=400, detail="Question is required")
    t0 = time.time()
    answer = rag_chain.invoke(req.question)
    sources = []
    if req.include_sources:
        docs = retriever.invoke(req.question)
        sources = list(set(d.metadata.get("source", "").split("/")[-1] for d in docs))
    return QueryResponse(answer=answer, sources=sources, latency_ms=round((time.time()-t0)*1000, 1))
'''

with open("api.py", "w") as f:
    f.write(api_code)

print("api.py written. Run: uvicorn api:app --reload --port 8000")
print("Docs at: http://localhost:8000/docs")